In [ ]:
import pandas as pd
import json
import os
import re
import numpy as np
np.random.seed(1029)
import warnings
from tqdm import tqdm
from datetime import timedelta
from difflib import SequenceMatcher
import re
warnings.filterwarnings('ignore')

In [ ]:
df_tornado_cash_address = pd.read_csv('Dataset/tornado_cash_address.csv')
address_to_name = dict(zip(df_tornado_cash_address['Address'], df_tornado_cash_address['Name Tag']))

In [ ]:
df_tornado_txns = pd.read_csv('Dataset/mixbroker_raw_data/Graph/ETH.csv')

address_set = set(df_tornado_txns['From'].tolist())
address_set.update(df_tornado_txns['To'].tolist())

address_to_name = {address: address_to_name[address] for address in address_set if address in address_to_name}
address_to_name, list(address_to_name.keys())

In [ ]:
from glob import glob
import pandas as pd

files = sorted(glob('Dataset/TornadoRelatedTransactions/Normal/*.csv'))
df_normal = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

files = sorted(glob('Dataset/TornadoRelatedTransactions/Internal/*.csv'))
df_internal = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

files = sorted(glob('Dataset/TornadoRelatedTransactions/ERC20/*.csv'))
df_erc20 = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

len(df_normal), len(df_internal), len(df_erc20)

In [ ]:
print('Total number of transactions:', len(set(df_normal['hash']) | set(df_internal['hash']) | set(df_erc20['hash'])))
print('Total number of from addresses:', len(set(df_normal['from']) | set(df_internal['from']) | set(df_erc20['from'])))
print('Total number of to addresses:', len(set(df_normal['to']) | set(df_internal['to']) | set(df_erc20['to'])))
print('Total number of unique addresses:', len(set(df_normal['from']) | set(df_internal['from']) | set(df_erc20['from']) | set(df_normal['to']) | set(df_internal['to']) | set(df_erc20['to'])))

In [ ]:
address_to_name = {
  '0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc': 'Tornado.Cash: 0.1 ETH',
  '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936': 'Tornado.Cash: 1 ETH',
  '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf': 'Tornado.Cash: 10 ETH',
  '0xa160cdab225685da1d56aa342ad8841c3b53f291': 'Tornado.Cash: 100 ETH',
  '0xd90e2f925da726b50c4ed8d0fb90ad053324f31b': 'Tornado.Cash: Router',
  '0x905b63fff465b9ffbf41dea908ceb12478ec7601': 'Tornado.Cash: Old Proxy',
  '0x722122df12d4e14e13ac3b6895a86e84145b6967': 'Tornado.Cash: Proxy'}

tornado_constractes = ['0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc',
                    '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936',
                    '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf',
                    '0xa160cdab225685da1d56aa342ad8841c3b53f291']

In [ ]:
# 创建几个文件
# 所有构图交易graph_transactions.csv
# 所有存入地址信息deposit_address_info.csv (包括存入时间、地址标签、存入方式)
# 所有取出地址信息withdraw_address_info.csv (包括取出时间、地址标签、取出方式、取出比例)

In [ ]:
# 查看混币输出的标签有哪些
with open('Dataset/tornado_cash_neighbor_address_label.json', 'r') as f:
    tornado_cash_neighbor_address_label = json.load(f)

print(len(tornado_cash_neighbor_address_label))
# Remove entries where both "label" and "title" are empty strings
tornado_cash_neighbor_address_label = {
    addr: info
    for addr, info in tornado_cash_neighbor_address_label.items()
    if info.get("label", "") != "Source Code" and (info.get("label", "") != "" or info.get("title", "") != "")
}
print(len(tornado_cash_neighbor_address_label))

# 获取所有router的带标签的存入地址
router_addresses = ['0xd90e2f925da726b50c4ed8d0fb90ad053324f31b','0x905b63fff465b9ffbf41dea908ceb12478ec7601','0x722122df12d4e14e13ac3b6895a86e84145b6967']
router_tx_df_list = []
for router_address in router_addresses:
    router_tx_df = pd.read_csv(f'Dataset/TornadoRelatedTransactions/Normal/{router_address}.csv',
        usecols=['blockNumber', 'contractAddress', 'from', 'gas', 'gasUsed',
            'hash', 'input', 'isError', 'timeStamp', 'to', 'value'])
    router_tx_df_list.append(router_tx_df)
router_tx_dfs = pd.concat(router_tx_df_list, ignore_index=True)
len(router_tx_dfs)

In [ ]:
def CreateGraphTransactionsFile(address):
    os.makedirs(f'Dataset/TornadoContractTransaction/{address}', exist_ok=True)

    # 所有提取交易都是内部交易
    internal_tx_raw_df = pd.read_csv(
        f'Dataset/TornadoRelatedTransactions/Internal/{address}.csv',
        usecols=['blockNumber', 'contractAddress', 'from', 'gas', 'gasUsed',
            'hash', 'input', 'isError', 'timeStamp', 'to', 'value']
    )
    internal_wd_tx_df = internal_tx_raw_df[internal_tx_raw_df['from']==address]
    # internal_wd_tx_df = internal_wd_tx_df[internal_wd_tx_df['value'] > threshold]
    internal_wd_tx_df = internal_wd_tx_df.drop_duplicates(subset=['hash'], keep='first')
    internal_wd_tx_df['tornado_opt'] = 'withdraw'

    # 判断是否是router取款
    router_tx_hashes = set(router_tx_dfs['hash'].tolist())
    internal_wd_tx_df['opt_type'] = internal_wd_tx_df['hash'].apply(
        lambda h: 'router' if h in router_tx_hashes else 'direct'
    )

    # 判断取款地址是否有标签
    internal_wd_tx_df['address_label'] = internal_wd_tx_df['to'].map(lambda addr: tornado_cash_neighbor_address_label.get(addr, {}).get('label', ''))
    internal_wd_tx_df['address_title'] = internal_wd_tx_df['to'].map(lambda addr: tornado_cash_neighbor_address_label.get(addr, {}).get('title', ''))

    internal_wd_tx_df.to_csv(f'Dataset/TornadoContractTransaction/{address}/withdraw.csv', index=False)
    # return internal_tx_df
    # 存入交易分为多种
    # 在Normal交易中直接存入
    normal_tx_raw_df = pd.read_csv(
        f'Dataset/TornadoRelatedTransactions/Normal/{address}.csv',
        usecols=['blockNumber', 'contractAddress', 'from', 'gas', 'gasUsed',
            'hash', 'input', 'isError', 'timeStamp', 'to', 'value']
    )
    normal_ds_tx_df = normal_tx_raw_df[normal_tx_raw_df['to']==address]
    normal_ds_tx_df['value'] = normal_ds_tx_df['value'].astype(float)
    normal_ds_tx_df = normal_ds_tx_df[normal_ds_tx_df['value']>0]
    normal_ds_tx_df['tornado_opt'] = 'deposit'
    normal_ds_tx_df['opt_type'] = 'direct'
    normal_ds_tx_df['address_label'] = normal_ds_tx_df['from'].map(lambda addr: tornado_cash_neighbor_address_label.get(addr, {}).get('label', ''))
    normal_ds_tx_df['address_title'] = normal_ds_tx_df['from'].map(lambda addr: tornado_cash_neighbor_address_label.get(addr, {}).get('title', ''))
    print(f'normal deposit {address} number: {len(normal_ds_tx_df)}')
    # 通过router存入
    internal_tx_to_df = internal_tx_raw_df[internal_tx_raw_df['to']==address]
    internal_tx_to_df = internal_tx_to_df[internal_tx_to_df['from'].isin(router_addresses)]
    router_deposit_hashes = internal_tx_to_df['hash'].tolist()
    internal_ds_tx_df = router_tx_dfs[router_tx_dfs['hash'].isin(router_deposit_hashes)]
    internal_ds_tx_df['address_label'] = internal_ds_tx_df['from'].map(lambda addr: tornado_cash_neighbor_address_label.get(addr, {}).get('label', ''))
    internal_ds_tx_df['address_title'] = internal_ds_tx_df['from'].map(lambda addr: tornado_cash_neighbor_address_label.get(addr, {}).get('title', ''))
    internal_ds_tx_df['tornado_opt'] = 'deposit'
    internal_ds_tx_df['opt_type'] = 'router'
    print(f'router deposit {address} number: {len(internal_ds_tx_df)}')
    # 保存成两个csv文件
    ds_tx_df = pd.concat([normal_ds_tx_df, internal_ds_tx_df])
    ds_tx_df.to_csv(f'Dataset/TornadoContractTransaction/{address}/deposit.csv', index=False)
    print(f'withdraw {address} number: {len(internal_wd_tx_df)}')
    print(f'deposit {address} number: {len(ds_tx_df)}')

tornado_constractes = ['0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc',
                    '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936',
                    '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf',
                    '0xa160cdab225685da1d56aa342ad8841c3b53f291']
for addr in tornado_constractes:
    print(addr, address_to_name[addr])
    CreateGraphTransactionsFile(addr)

In [ ]:
# 找到存入和提取地址的标签匹配
def GetTornadoCashPairAddresses(address, label_type='all', heist_type='hack', tx_duplicate_check=False, mode='val'):
    # 获取所有存入和提取地址
    ds_tx_df = pd.read_csv(f'Dataset/TornadoContractTransaction/{address}/deposit.csv')
    wd_tx_df = pd.read_csv(f'Dataset/TornadoContractTransaction/{address}/withdraw.csv')
    
    ds_tx_df = ds_tx_df[(ds_tx_df['address_title'].notna())]
    wd_tx_df = wd_tx_df[(wd_tx_df['address_title'].notna())]
    
    matches = []
    neg_matches = []
    wd_row_set = set()
    addr_set = set()
    ds_row_set = set()

    for i in range(len(ds_tx_df)):
        ds_address = ds_tx_df.iloc[i]['from']
        ds_hash = ds_tx_df.iloc[i]['hash']
        ds_blockNumber = ds_tx_df.iloc[i]['blockNumber']
        ds_label = ds_tx_df.iloc[i]['address_label']
        if label_type == 'heist':
            if ('heist' not in ds_label.lower()) & ('exploit' not in ds_label.lower()) & ('hack' not in ds_label.lower()) & ('fake' not in ds_label.lower()):
                continue
        
        if 'Tornado.Cash: Donate' in ds_label or 'Fee Recipient' in ds_label:
            continue
        
        ds_title = ds_tx_df.iloc[i]['address_label']
        ds_timeStamp = ds_tx_df.iloc[i]['timeStamp']
        wd_tx_df_query = wd_tx_df[(wd_tx_df['timeStamp']>ds_timeStamp)&(wd_tx_df['timeStamp']<ds_timeStamp+90*24*60*60)]

        if ('Fake_Phishing' in ds_label) and (heist_type == 'all'):
            if mode == 'val':
                wd_tx_df_query = wd_tx_df[(wd_tx_df['timeStamp']>ds_timeStamp)&(wd_tx_df['timeStamp']<ds_timeStamp+7*24*60*60)]
                def is_nei_num(a, b, threshold=1):
                    def extract_phishing_number(s):
                        match = re.search(r'Fake_Phishing(\d+);', str(s))
                        return int(match.group(1)) if match else None
                    if extract_phishing_number(a) is None or extract_phishing_number(b) is None:
                        return False
                    return abs(extract_phishing_number(a)-extract_phishing_number(b)) <= threshold
                    
                wd_tx_df_query = wd_tx_df_query[
                    wd_tx_df_query['address_label'].apply(lambda x: is_nei_num(x, ds_label))
                ]
            else:
                wd_tx_df_query = wd_tx_df_query[
                    wd_tx_df_query['address_label'] == ds_title
                ]
        else:
            # def is_similar(a, b, threshold=0.2):
            #     if pd.isna(a) or pd.isna(b):
            #         return False
            #     if str(a).split(' ')[0] == str(b).split(' ')[0]:
            #         return SequenceMatcher(None, str(a), str(b)).ratio() >= threshold
            #     else:
            #         return False

            # # 相似
            # wd_tx_df_query = wd_tx_df_query[
            #     wd_tx_df_query['address_title'].apply(lambda x: is_similar(x, ds_title))
            # ]
            # # 相同 ———— 训练集
            if mode == 'train':
                wd_tx_df_query = wd_tx_df_query[
                    wd_tx_df_query['address_label'] == ds_title
                ]
            else:
                # 相似  ———— 验证集
                ds_title = ds_tx_df.iloc[i]['address_label']
                ds_title_split = ' '.join(ds_tx_df.iloc[i]['address_label'].split(' ')[:-1])
                if ds_title_split == '':
                    ds_title_split = ds_title
                ds_timeStamp = ds_tx_df.iloc[i]['timeStamp']
                # 找到地址相同title或者非常相似的title
                wd_tx_df_query = wd_tx_df_query[
                    wd_tx_df_query['address_label'].fillna('').str.startswith(str(ds_title_split))
                ]

        if len(wd_tx_df_query)>0:
            for wd_idx, wd_row in wd_tx_df_query.iterrows():
                if mode == 'val':
                    if ds_address == wd_row['to']:
                        continue
                match = {
                    'deposit_address': ds_address,
                    'deposit_hash': ds_hash,
                    'deposit_label': ds_label,
                    'deposit_title': ds_tx_df.iloc[i]['address_title'],
                    'deposit_opt_type': ds_tx_df.iloc[i]['opt_type'],
                    'deposit_blockNumber': int(ds_blockNumber),
                    'deposit_timeStamp': int(ds_timeStamp),
                    'deposit_datetime': pd.to_datetime(int(ds_timeStamp), unit='s').strftime('%Y-%m-%d %H:%M:%S'),
                    'withdraw_address': wd_row['to'],
                    'withdraw_hash': wd_row['hash'],
                    'withdraw_label': wd_row['address_label'],
                    'withdraw_title': wd_row['address_title'],
                    'withdraw_opt_type': wd_row['opt_type'],
                    'withdraw_blockNumber': int(wd_row['blockNumber']),
                    'withdraw_timeStamp': int(wd_row['timeStamp']),
                    'withdraw_datetime': pd.to_datetime(int(wd_row['timeStamp']), unit='s').strftime('%Y-%m-%d %H:%M:%S'),
                    'contract_address': address,
                    'label': True
                }
                
                if wd_row['hash'] not in wd_row_set:
                    if tx_duplicate_check:
                        wd_row_set.add(wd_row['hash'])
                    if ds_address not in addr_set:
                        addr_set.add(ds_address)    
                        matches.append(match)
                    break
        else:
            if np.random.rand() >= 0:
                continue
            wd_tx_df_query = wd_tx_df[(wd_tx_df['timeStamp']>ds_timeStamp)&(wd_tx_df['timeStamp']<ds_timeStamp+7*24*60*60)]
            if len(neg_matches) >= len(matches):
                continue
            if 'Fake_Phishing' in ds_tx_df.iloc[i]['address_label']:
                continue

            # 随机选一个withdraw中是黑客攻击，但是前缀不同的作为负样本
            if mode == 'val' and len(wd_tx_df_query) > 0:
                # 取deposit列当前黑客label，找到所有withdraw中label为heist的（黑客攻击），且address_label前缀不同
                cur_ds_title_split = ' '.join(ds_tx_df.iloc[i]['address_label'].split(' ')[:-1])
                # print(wd_tx_df_query['address_label'])
                heist_neg_candidates = wd_tx_df_query[
                    (wd_tx_df_query['address_label'].notna()) &
                    (wd_tx_df_query['address_label'].str.contains('Exploit') | wd_tx_df_query['address_label'].str.contains('Hacker') | wd_tx_df_query['address_label'].str.contains('Heist')) &
                    (~wd_tx_df_query['address_label'].str.contains('Fake_Phishing')) &
                    (~wd_tx_df_query['address_label'].str.contains('Tornado.Cash')) &
                    (~wd_tx_df_query['address_label'].str.startswith(str(cur_ds_title_split)))
                ]
                
                if len(heist_neg_candidates) > 0:
                    neg_row = heist_neg_candidates.sample(n=1, random_state=np.random.randint(0, 1_000_000)).iloc[0]
                    neg_match = {
                        'deposit_address': ds_address,
                        'deposit_hash': ds_hash,
                        'deposit_label': ds_label,
                        'deposit_title': ds_tx_df.iloc[i]['address_title'],
                        'deposit_opt_type': ds_tx_df.iloc[i]['opt_type'],
                        'deposit_blockNumber': int(ds_blockNumber),
                        'deposit_timeStamp': int(ds_timeStamp),
                        'deposit_datetime': pd.to_datetime(int(ds_timeStamp), unit='s').strftime('%Y-%m-%d %H:%M:%S'),
                        'withdraw_address': neg_row['to'],
                        'withdraw_hash': neg_row['hash'],
                        'withdraw_label': neg_row['address_label'],
                        'withdraw_title': neg_row['address_title'],
                        'withdraw_opt_type': neg_row['opt_type'],
                        'withdraw_blockNumber': int(neg_row['blockNumber']),
                        'withdraw_timeStamp': int(neg_row['timeStamp']),
                        'withdraw_datetime': pd.to_datetime(int(neg_row['timeStamp']), unit='s').strftime('%Y-%m-%d %H:%M:%S'),
                        'contract_address': address,
                        'label': False
                    }
                    neg_matches.append(neg_match)
    # if matches:
    #     with open(f'Dataset/AMLValidation/{mode}_{label_type}.json', 'w') as f: # _{address}
    #         f.write(json.dumps(matches))
    
    return matches, neg_matches


mode = 'train'
label_type = 'all'
heist_type = 'all'
saved_matches = []

for addr in tornado_constractes:
    matches, neg_matches= GetTornadoCashPairAddresses(addr, label_type=label_type, mode=mode, heist_type=heist_type, tx_duplicate_check=False)
    
    print(addr, address_to_name[addr], len(matches), len(neg_matches))
    saved_matches.extend(matches)
    saved_matches.extend(neg_matches)

# 训练集去重，保留deposit_address和withdraw_address唯一的一组
deduped = {
    (m['deposit_address'], m['withdraw_address']): m
    for m in saved_matches
    if not (isinstance(m.get('deposit_label'), str) and m['deposit_label'].endswith('.eth'))
}
print(len(deduped))
with open(f'Dataset/AMLValidation/{mode}_{label_type}_{heist_type}.json', 'w') as f:
    json.dump(list(deduped.values()), f, indent=4)


mode = 'val'
label_type = 'heist'
heist_type = 'all'
saved_matches = []

for addr in tornado_constractes:
    matches, neg_matches= GetTornadoCashPairAddresses(addr, label_type=label_type, mode=mode, heist_type=heist_type, tx_duplicate_check=False)
    
    print(addr, address_to_name[addr], len(matches), len(neg_matches))
    saved_matches.extend(matches)
    saved_matches.extend(neg_matches)

# 训练集去重，保留deposit_address和withdraw_address唯一的一组
deduped = {
    (m['deposit_address'], m['withdraw_address']): m
    for m in saved_matches
    if not (isinstance(m.get('deposit_label'), str) and m['deposit_label'].endswith('.eth'))
}
print(len(deduped))
with open(f'Dataset/AMLValidation/{mode}_{label_type}_{heist_type}.json', 'w') as f:
    json.dump(list(deduped.values()), f, indent=4)

In [ ]:
def getLabeledCount(address):
    deposit_df = pd.read_csv(f'Dataset/TornadoContractTransaction/{address}/deposit.csv')
    withdraw_df = pd.read_csv(f'Dataset/TornadoContractTransaction/{address}/withdraw.csv')
    deposit_df = deposit_df[deposit_df['address_title'].apply(lambda x: isinstance(x, str) and not x.endswith('.eth') and x.strip() != '')]
    withdraw_df = withdraw_df[withdraw_df['address_title'].apply(lambda x: isinstance(x, str) and not x.endswith('.eth') and x.strip() != '')]
    return len(deposit_df), len(withdraw_df)

total_deposit = 0
total_withdraw = 0
for addr in tornado_constractes:
    deposit_cnt, withdraw_cnt = getLabeledCount(addr)
    print(addr, (deposit_cnt, withdraw_cnt))
    total_deposit += deposit_cnt
    total_withdraw += withdraw_cnt
print("总deposit数:", total_deposit)
print("总withdraw数:", total_withdraw)

In [ ]:

with open('Dataset/AMLValidation/train_all_all.json', 'r') as f:
    train_data = json.load(f)


from collections import Counter

# 统计deposit_label类别的数量（每个分号分割后不同类别最多统计一次）
label_counter = Counter()
for m in train_data:
    deposit_label = m.get('deposit_label', '')
    if isinstance(deposit_label, str):
        # 以分号切分，并strip去掉空白
        unique_labels = set(lbl.strip() for lbl in deposit_label.split(';') if lbl.strip())
        for lbl in unique_labels:
            label_counter[lbl] += 1

print("分号切分后的deposit_label类别统计:")
for label, cnt in label_counter.most_common():
    print(f"{label}: {cnt}")

In [ ]:
tornado_constractes = ['0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc',
                    '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936',
                    '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf',
                    '0xa160cdab225685da1d56aa342ad8841c3b53f291']

heist_label_dict = {}
heist_label_dict_unique = {}

for addr in tornado_constractes:
    print(addr)
    deposit_df = pd.read_csv(f'Dataset/TornadoContractTransaction/{addr}/withdraw.csv')

    deposit_labels = [
        {addr: label_info['label']}
        for addr in set(deposit_df['to'])
        if (label_info := tornado_cash_neighbor_address_label.get(addr))
        and any(keyword in label_info['label'].lower() for keyword in ['heist', 'exploit', 'hack', 'fake'])
        and '.eth' not in label_info['label']
        # and 'Fake_Phishing' not in label_info['label']
    ]

    for d in deposit_labels:
        heist_label_dict.update(d)
    
    print(len(deposit_labels))

    def clean_label(label):
        # Remove trailing numbers from each part
        parts = label.split(';')
        cleaned_parts = []
        seen = set()
        for part in parts:
            cleaned = re.sub(r'\s*\d+$', '', part.strip())
            if cleaned not in seen:
                cleaned_parts.append(cleaned)
                seen.add(cleaned)
        return ';'.join(cleaned_parts)

    # Build a mapping from cleaned label to (address, original label, address int for comparison)
    label_map = {}
    for addr in set(deposit_df['to']):
        label_info = tornado_cash_neighbor_address_label.get(addr)
        if not label_info:
            continue
        label = label_info['label']
        if not any(keyword in label.lower() for keyword in ['heist', 'exploit', 'hacker']):
            continue
        cleaned = clean_label(label)
        # Use int value of address for comparison (lower is preferred)
        addr_int = int(addr, 16)
        if cleaned not in label_map or addr_int < label_map[cleaned][2]:
            label_map[cleaned] = (addr, cleaned, addr_int)

    deposit_labels_cleaned = [{addr: cleaned} for addr, cleaned, _ in label_map.values()]
    # Save as a key-value mapping instead of a list
    for d in deposit_labels_cleaned:
        heist_label_dict_unique.update(d)
    

    print(len(deposit_labels_cleaned))
# with open('Dataset/tornado_heist.json', 'w') as f:
#     json.dump(heist_label_dict, f)
# with open('Dataset/tornado_heist_unique.json', 'w') as f:
#     json.dump(heist_label_dict_unique, f)
print(len(heist_label_dict),len(heist_label_dict_unique))

In [ ]:
# 获取待检测的存入洗钱交易/地址
tornado_constractes = ['0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc',
                    '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936',
                    '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf',
                    '0xa160cdab225685da1d56aa342ad8841c3b53f291']

heist_validation = []

for constract_addr in tornado_constractes:
    print(constract_addr)
    deposit_df = pd.read_csv(f'Dataset/TornadoContractTransaction/{constract_addr}/deposit.csv')
    deposit_df = deposit_df.drop_duplicates(subset=['from'], keep='first')
    for idx, row in deposit_df.iterrows():
        addr_deposit = row['from']
        label_info = tornado_cash_neighbor_address_label.get(addr_deposit)
        if not label_info:
            continue
        label = label_info['label']
        if not any(keyword in label.lower() for keyword in ['heist', 'exploit', 'hack', 'fake']):
            continue
        if '.eth' in label and 'Fake_Phishing' not in label:
            continue
        # if 'Fake_Phishing' in label:
        #     continue
        deposit_time = row['timeStamp'] if 'timeStamp' in row else None
        heist_validation.append({
            'deposit_address': addr_deposit,
            'label': label,
            'deposit_time': deposit_time,
            'contract_address': constract_addr,
            'opt_type': row['opt_type']
        })

with open('Dataset/AMLValidation/heist_validation_all.json', 'w') as f:
    json.dump(heist_validation, f, indent=4)

print(len(heist_validation))

agg = {}
for i in heist_validation:
    a = i['deposit_address']
    t = int(i.get('deposit_time', 0) or 0)
    if a not in agg:
        agg[a] = {'deposit_address':a, 'label':i['label'], 'deposit_time':t, 'contract_address':[i['contract_address']], 'opt_type':[i['opt_type']]}
    else:
        if i['contract_address'] not in agg[a]['contract_address']:
            agg[a]['contract_address'].append(i['contract_address'])
            agg[a]['opt_type'].append(i['opt_type'])
        if t < agg[a]['deposit_time']:
            agg[a]['deposit_time'] = t
heist_validation_uniq = list(agg.values())
with open('Dataset/AMLValidation/heist_validation_all_uniq.json', 'w') as f:
    json.dump(heist_validation_uniq, f, indent=4)
print(len(heist_validation_uniq))



In [ ]:
# 修改已有标签数据集的格式成json
def ReadCSVTransactions(addresses):   # 读取混币合约地址列表所有的交易
    tx_all_list = []

    for addr, name in address_to_name.items():
        if addr not in addresses:
            continue
        for tx_type in ['Normal', 'Internal']:
            data_df = pd.read_csv(f'Dataset/TornadoRelatedTransactions/{tx_type}/{addr}.csv',
            usecols=['blockNumber', 'contractAddress', 'from', 'gas', 'gasUsed',
                'hash', 'input', 'isError', 'timeStamp', 'to', 'value'])
            data_df['contract_address'] = addr
            data_df['opt_type'] = 'direct'
            tx_all_list.append(data_df)
            router_tx_dfs_query = router_tx_dfs[router_tx_dfs['hash'].isin(data_df['hash'])]
            if len(router_tx_dfs_query)>0:
                router_tx_dfs_query['contract_address'] = addr
                router_tx_dfs_query['opt_type'] = 'router'
                tx_all_list.append(router_tx_dfs_query)
            
    tx_all_df = pd.concat(tx_all_list, ignore_index=True)
    return tx_all_df

In [ ]:
ens_data_df = pd.read_csv('Dataset/mixbroker_raw_data/Graph/ens_edge.csv')
tx_all_df = ReadCSVTransactions(['0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936'])

for i in range(len(ens_data_df)):
    sender = ens_data_df.iloc[i]['Sender']
    receiver = ens_data_df.iloc[i]['Receiver']
    
    sender_tx_df = tx_all_df[tx_all_df['from']==sender].drop_duplicates(subset=['hash'], keep='first')
    receiver_tx_df = tx_all_df[tx_all_df['to']==receiver].drop_duplicates(subset=['hash'], keep='first')
    print(sender, receiver, len(sender_tx_df), len(receiver_tx_df))
    if (len(sender_tx_df)>0)&(len(receiver_tx_df)>0):
        print(sender_tx_df.iloc[0]['hash'], receiver_tx_df.iloc[0]['hash'])

In [ ]:
days = [7, 30, 90, 365, 730]
jianchu = []
weiman5 = []
man5 = []
heike = []
youbiao = []
wubiao = []

for day in days:
    print(day)
    with open(f'Results/link_predictions_hack_{day}day_top5.json', 'r') as f:
        heist_validation_result = json.load(f)

    cnt_empty = 0
    heist_one = 0
    unknown_one = 0
    heist_total = 0
    with_all = 0
    empty_label = 0
    top5_cnt = 0
    top0_5_cnt = 0

    for hack in heist_validation_result:
        if len(hack['top_n_withdraw_addresses']) == 0:
            cnt_empty += 1
        # print(hack['top_n_withdraw_addresses'])
        local_heist_total = 0
        with_all += len(hack['top_n_withdraw_addresses'])
        if (len(hack['top_n_withdraw_addresses']) < 5) and (len(hack['top_n_withdraw_addresses']) > 0):
            top0_5_cnt += 1
        if len(hack['top_n_withdraw_addresses']) == 5:
            top5_cnt += 1
        for withdraw in hack['top_n_withdraw_addresses']:
            if 'heist' in withdraw['withdraw_address_label'].lower() or 'exploit' in withdraw['withdraw_address_label'].lower() or 'hack' in withdraw['withdraw_address_label'].lower() or 'fake' in withdraw['withdraw_address_label'].lower():
                heist_total += 1
                local_heist_total += 1
            elif withdraw['withdraw_address_label'] == '':
                empty_label += 1
            else:
                unknown_one +=1
        if local_heist_total >= 1:
            heist_one += 1
    print("检出率",1-(cnt_empty / len(heist_validation_result)), "未满5率", top0_5_cnt / len(heist_validation_result), "满5率", top5_cnt / len(heist_validation_result))   # 7day 14day 30day 60day 90day
    print("黑客率", heist_total / with_all, "有标签但未知率", unknown_one / with_all, "空标签率", empty_label / with_all)   # top 1\3\5

    jianchu.append(1-(cnt_empty / len(heist_validation_result)))
    weiman5.append(top0_5_cnt / len(heist_validation_result))
    man5.append(top5_cnt / len(heist_validation_result))
    heike.append(heist_total / with_all)
    youbiao.append(unknown_one / with_all)
    wubiao.append(empty_label / with_all)

    summary_data = {
        "days": days,
        "jianchu": jianchu,
        "weiman5": weiman5,
        "man5": man5,
        "heike": heike,
        "youbiao": youbiao,
        "wubiao": wubiao
    }
    with open('Results/heist_eval_summary.json', 'w') as f:
        json.dump(summary_data, f, ensure_ascii=False, indent=2)


In [ ]:
# 计算标签地址中黑客地址的数量
len(tornado_cash_neighbor_address_label), len([label_info['label'] for label_info in tornado_cash_neighbor_address_label.values() if 'heist' in label_info['label'].lower() or 'exploit' in label_info['label'].lower() or 'hack' in label_info['label'].lower() or 'fake' in label_info['label'].lower()])

In [ ]:
rule2_files = ['Dataset/tornado_raw_data/heuristic2Mixer_0.1ETH.csv', 'Dataset/tornado_raw_data/heuristic2Mixer_1ETH.csv', 'Dataset/tornado_raw_data/heuristic2Mixer_10ETH.csv', 'Dataset/tornado_raw_data/heuristic2Mixer_100ETH.csv']
rule3_files = ['Dataset/tornado_raw_data/heuristic3Mixer_0.1ETH.csv', 'Dataset/tornado_raw_data/heuristic3Mixer_1ETH.csv', 'Dataset/tornado_raw_data/heuristic3Mixer_10ETH.csv', 'Dataset/tornado_raw_data/heuristic3Mixer_100ETH.csv']
# 获取 rule2 和 rule3 的去重数据量
import pandas as pd

# 合并所有 rule2、rule3 数据，基于 sender+receiver 去重
rule2_df = pd.concat([pd.read_csv(f) for f in rule2_files], ignore_index=True)
rule3_df = pd.concat([pd.read_csv(f) for f in rule3_files], ignore_index=True)

rule2_unique = rule2_df.drop_duplicates(subset=['sender', 'receiver'])
rule3_unique = rule3_df.drop_duplicates(subset=['sender', 'receiver'])

print("rule2 去重后数据量:", len(rule2_unique))
print("rule3 去重后数据量:", len(rule3_unique))



In [ ]:
with open('Dataset/AMLValidation/val_heist_all.json', 'r') as f:
    test_pairs = json.load(f)
    
test_pairs = list({(p['deposit_address'], p['withdraw_address']): p for p in test_pairs}.values())
test_pairs.sort(key=lambda p: (p['deposit_address'], p['withdraw_address']))

with open('Dataset/AMLValidation/val_heist_all_12.json', 'w') as f:
    json.dump(test_pairs, f, indent=4)
test_pairs_deposits = [p['deposit_address'] for p in test_pairs]

In [ ]:
link_predictions_res = json.load(open('Results/link_predictions_hack_phish_90day_top10.json', 'r'))
link_predictions_res

deposit_addresses = []
deposit_address_map = {p['deposit_address']: p for p in link_predictions_res if p['deposit_address'] in test_pairs_deposits}
for addr in test_pairs_deposits:
    if addr in deposit_address_map:
        deposit_addresses.append(deposit_address_map[addr])
with open('Dataset/AMLValidation/val_heist_all_result_90.json', 'w') as f:
    json.dump(deposit_addresses, f, indent=4)
deposit_addresses

In [ ]:
import numpy as np

def compute_metrics(predictions, ground_truth_pairs, k_list=[1, 5, 10]):
    gt_withdraw_map = {p['deposit_address']: p['withdraw_address'] for p in ground_truth_pairs}

    MRRs = []
    hits_at_k = {k: 0 for k in k_list}
    recalls_at_k = {k: 0 for k in k_list}
    AP_at_k = {k: [] for k in k_list}  # For MAP@k
    count = 0

    for pred in predictions:
        daddr = pred['deposit_address']
        top_wds = [wd['withdraw_address'] for wd in pred.get('top_n_withdraw_addresses', [])]
        if daddr not in gt_withdraw_map:
            continue
        true_wd = gt_withdraw_map[daddr]
        count += 1
        hit = False
        ap_list = []
        # MRR
        for rank, wd in enumerate(top_wds):
            if wd == true_wd:
                MRRs.append(1.0 / (rank + 1))
                hit = True
                break
        if not hit:
            MRRs.append(0.0)
        # Hit@k, Recall@k, AP@k
        for k in k_list:
            in_topk = true_wd in top_wds[:k]
            hits_at_k[k] += int(in_topk)
            recalls_at_k[k] += int(in_topk)
            # For AP@k: compute precision at position of correct withdraw if in topk, else 0
            if in_topk:
                rank = top_wds[:k].index(true_wd)
                precision = 1.0 / (rank + 1)
                AP_at_k[k].append(precision)
            else:
                AP_at_k[k].append(0.0)

    metrics = {
        "MRR": np.mean(MRRs) if MRRs else 0.0
    }
    for k in k_list:
        metrics[f"Hit@{k}"] = hits_at_k[k] / count if count else 0.0
        metrics[f"Recall@{k}"] = recalls_at_k[k] / count if count else 0.0
        metrics[f"MAP@{k}"] = np.mean(AP_at_k[k]) if AP_at_k[k] else 0.0
    return metrics

# 计算指标
with open('Dataset/AMLValidation/val_heist_all_result_365.json') as f:
    preds = json.load(f)
with open('Dataset/AMLValidation/val_heist_all.json') as f:
    gt_pairs = json.load(f)

metrics = compute_metrics(preds, gt_pairs, k_list=[1, 5, 10])
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")



In [ ]:
import pandas as pd


for contract_addr in ['0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc', '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936', '0x910cbd523d972eb0a6f4cae4618ad62622b39dbf', '0xa160cdab225685da1d56aa342ad8841c3b53f291']:
    # 读取withdraw.csv文件
    df = pd.read_csv(f"Dataset/TornadoContractTransaction/{contract_addr}/withdraw.csv")

    # 检查timeStamp字段，部分csv文件时间戳为int（Unix时间戳），部分为字符串，需做兼容
    # 先尝试解析为int型，再转换成datetime
    def parse_timestamp(x):
        try:
            return pd.to_datetime(int(x), unit="s")
        except Exception:
            try:
                return pd.to_datetime(x)
            except Exception:
                return pd.NaT

    df['parsed_time'] = df['timeStamp'].apply(parse_timestamp)
    df = df.dropna(subset=['parsed_time'])

    # 按'year-month'进行分组计数
    df['year_month'] = df['parsed_time'].dt.to_period('M')
    withdraw_per_month = df.groupby('year_month').size()

    # 计算月均取出交易数量
    monthly_avg = withdraw_per_month.mean()
    print(f"合约{contract_addr}平均每个月有{monthly_avg:.2f}笔取出交易")
